# News Topic Classification for Short Headlines

**ARTI-404 | Group 5 | Option 3 — News Topic Classification**  
**Research question:** Which method performs better on very short headlines (six words or fewer): **TF-IDF + Logistic Regression** or **pretrained MiniLM sentence embeddings + Logistic Regression**?

This is the **team's main, reproducible notebook**. Both models were trained on the same prepared training split, selected with the validation split, and frozen before evaluating on the held-out test split. We use the provided frozen models and predictions for reproducible final evaluation; this notebook does **not** retrain models or tune hyperparameters on the test data.

**Notebook sequence:** Setup → Data → Method 1 → Method 2 → Results → Error analysis → Try it → Sources.

## 1. Setup

Run `python -m pip install -r requirements.txt` in your environment before **Kernel → Restart & Run All**. Start Jupyter in the project root. Local test and saved predictions are included in the submission ZIP; regenerating the dataset needs Internet access. MiniLM live prediction additionally downloads pretrained weights on first use, while the default TF-IDF Try it works offline.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import joblib
from IPython.display import display
from evaluation import evaluate, LABEL_NAMES
from predict_news import predict_tfidf, predict_minilm

ROOT = Path.cwd()
assert (ROOT / "data/test.csv").exists(), "Run Jupyter from the project root and provide data/test.csv"
assert (ROOT / "outputs/tfidf/tfidf_logreg.joblib").exists(), "Missing the frozen TF-IDF model"
assert (ROOT / "results/minilm_predictions.csv").exists(), "Run python prepare_minilm_predictions.py first"
print("Project directory:", ROOT.resolve())
print("Models: TF-IDF frozen pipeline + MiniLM pretrained embeddings with frozen classifier")

Project directory: C:\Users\abdul\Downloads\NLP-Project-Git
Models: TF-IDF frozen pipeline + MiniLM pretrained embeddings with frozen classifier


## 2. Data

**Dataset:** [AG News](https://huggingface.co/datasets/sh0416/ag_news) (English news headlines; four classes). The original source contains 120,000 training and 7,600 test cases. Preprocessing conservatively normalizes whitespace, excludes ambiguous/duplicate headlines, and removes headline overlap. A stratified 10% validation split is taken *only* from the original training portion. All methods use the original headline `text` field and the same prepared splits.

The project defines **very short** as six whitespace-separated words or fewer.

In [2]:
train = pd.read_csv(ROOT / "data/train.csv")
val = pd.read_csv(ROOT / "data/val.csv")
test = pd.read_csv(ROOT / "data/test.csv")
assert all({'text','label','word_count','is_short'}.issubset(df.columns) for df in (train,val,test))
print("Train:", len(train), "Validation:", len(val), "Test:", len(test))
print("Short test headlines:", int(test.is_short.sum()))
print("Class mapping:", LABEL_NAMES)
counts = pd.concat([f.label.value_counts().sort_index() for f in (train,val,test)],axis=1)
counts.columns=['Train','Validation','Test']
counts.index=[LABEL_NAMES[i] for i in counts.index]
display(counts)

def keys(df):
    return set(df.text.astype(str).str.replace(r"\s+", " ", regex=True).str.strip().str.casefold())
assert not keys(train)&keys(val)
assert not keys(train)&keys(test)
assert not keys(val)&keys(test)
print("PASS: No normalized headline overlap across splits.")

Train: 101043 Validation: 11227 Test: 7464
Short test headlines: 3554
Class mapping: {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}


,Train,Validation,Test
World,25699,2856,1882
Sports,26028,2892,1886
Business,24922,2769,1844
Sci/Tech,24394,2710,1852


PASS: No normalized headline overlap across splits.


## 3. Method 1 — TF-IDF + Logistic Regression

A word-level TF-IDF vectorizer transforms headlines into sparse frequency-based features, using sublinear term frequency. A one-vs-rest logistic-regression classifier learns the four topic labels. Settings were compared on the **validation** data only, with macro F1 as the selection metric. See `traditional_model.py` and `outputs/tfidf/hyperparameter_comparison.csv` for training details.

In [3]:
tfidf_model = joblib.load(ROOT / "outputs/tfidf/tfidf_logreg.joblib")  # trusted team-provided artifact
info = json.loads((ROOT / "outputs/tfidf/model_info.json").read_text(encoding="utf-8"))
print("Selected settings:", info["selected_settings"])
display(pd.read_csv(ROOT / "outputs/tfidf/hyperparameter_comparison.csv"))
print("TF-IDF test prediction example:", LABEL_NAMES[int(tfidf_model.predict(["Team wins football final"])[0])])

Selected settings: {'ngram_range': [1, 1], 'C': 8.0}


,ngram_range,C,accuracy,macro_f1
0,"(1, 1)",8.0,0.878507,0.877722
1,"(1, 2)",8.0,0.876993,0.876115
2,"(1, 1)",2.0,0.876726,0.875735
3,"(1, 2)",2.0,0.875212,0.874175
4,"(1, 1)",0.5,0.862831,0.861602
5,"(1, 2)",0.5,0.860693,0.859394


TF-IDF test prediction example: Sports


## 4. Method 2 — Pretrained MiniLM sentence embeddings + Logistic Regression

We use pretrained `sentence-transformers/all-MiniLM-L6-v2` to produce normalized 384-dimensional embeddings, then train a **separate** multinomial logistic-regression classifier on the training data. Hyperparameters were selected on validation, not test. The classifier's coefficients are exported as portable NumPy arrays in `models/minilm_logreg_coefficients.npz`. The saved test predictions come from the original MiniLM training experiment; the adapter restores the source test-row order, original headlines and the **1–4** class IDs.

Source: `pretrained_model.py`, `results/metrics_B.json`, and `prepare_minilm_predictions.py`. The pretrained weights download automatically when running MiniLM live prediction.

In [4]:
weights = np.load(ROOT / 'models/minilm_logreg_coefficients.npz')
print('MiniLM embedding model:', (ROOT/'models/embedder_name.txt').read_text().strip())
print('Classifier weight dimensions:', weights['coef'].shape)
print('MiniLM class IDs (internal 0-based):', weights['classes'].tolist())
print('Saved MiniLM test predictions:', len(pd.read_csv(ROOT/'results/minilm_predictions.csv')))

MiniLM embedding model: sentence-transformers/all-MiniLM-L6-v2
Classifier weight dimensions: (4, 384)
MiniLM class IDs (internal 0-based): [0, 1, 2, 3]
Saved MiniLM test predictions: 7464


## 5. Results — Same held-out test data for both models

Final metrics use the same **7,464** test headlines, including **3,554** very-short headlines. `evaluation.py` checks each prediction file against the ordered original test text before calculating Accuracy and **macro F1**. Test results must never be used to change model settings.

In [5]:
metrics, errors = evaluate(
    test_path=ROOT / 'data/test.csv',
    tfidf_path=ROOT / 'results/tfidf_predictions.csv',
    minilm_path=ROOT / 'results/minilm_predictions.csv',
    output_dir=ROOT / 'results'
)
display(metrics.assign(accuracy_pct=lambda d: 100*d.accuracy,
                       macro_f1_pct=lambda d: 100*d.macro_f1).round(3))
for prefix in ('tfidf','minilm'):
    print(f'\n{prefix.upper()} confusion matrix (test set)')
    display(pd.read_csv(ROOT / 'results' / f'{prefix}_confusion_matrix.csv', index_col=0))

                       model          subset    n  accuracy  macro_f1
TF-IDF + Logistic Regression             all 7464    0.8829    0.8823
TF-IDF + Logistic Regression short_6_or_less 3554    0.8672    0.8656
MiniLM + Logistic Regression             all 7464    0.8382    0.8375
MiniLM + Logistic Regression short_6_or_less 3554    0.8106    0.8095

Error categories: {'both_correct': 5966, 'minilm_only_wrong': 624, 'both_wrong': 584, 'tfidf_only_wrong': 290}
Results saved: C:\Users\abdul\Downloads\NLP-Project-Git\results


,model,subset,n,accuracy,macro_f1,accuracy_pct,macro_f1_pct
0,TF-IDF + Logistic Regression,all,7464,0.883,0.882,88.290,88.226
1,TF-IDF + Logistic Regression,short_6_or_less,3554,0.867,0.866,86.719,86.556
2,MiniLM + Logistic Regression,all,7464,0.838,0.838,83.816,83.751
3,MiniLM + Logistic Regression,short_6_or_less,3554,0.811,0.809,81.064,80.945



TFIDF confusion matrix (test set)


,predicted_World,predicted_Sports,predicted_Business,predicted_Sci/Tech
actual_World,1668,80,86,48
actual_Sports,42,1787,32,25
actual_Business,77,58,1553,156
actual_Sci/Tech,79,55,136,1582



MINILM confusion matrix (test set)


,predicted_World,predicted_Sports,predicted_Business,predicted_Sci/Tech
actual_World,1608,97,110,67
actual_Sports,71,1701,71,43
actual_Business,92,88,1429,235
actual_Sci/Tech,85,51,198,1518


### Interpretation

On both the complete test set and the very-short subset, **TF-IDF + Logistic Regression performs better than the pretrained MiniLM embedding + Logistic Regression baseline**. These findings describe the two specific frozen classifiers and the approved evaluation; they do **not** imply that traditional features always outperform pretrained methods. Possible explanations include the strong association between topic words and labels in this dataset, the short context, and differences in representation and optimization.

Both methods have lower scores on very-short headlines than on the full set, consistent with the challenge posed by missing context.

## 6. Error analysis — Three real held-out cases

Each row is a genuine prediction mismatch against the **reference dataset label**. Some labels are debatable or semantically ambiguous; an error against the reference label is not proof that a reader would classify the headline the same way. Consult `docs/error_analysis.md` for a contextual discussion and limitations.

In [6]:
headlines = [
    'Live: Olympics day four',
    'Google Lowers Its IPO Price Range',
    'Small computers can have multiple personalities, too',
]
samples = errors.loc[errors.text.isin(headlines),
                     ['text','word_count','label_name','tfidf_prediction_name',
                      'minilm_prediction_name','error_pattern']]
assert len(samples)==3, 'Expected three traceable error cases from the held-out test set'
display(samples.reset_index(drop=True))
print('All errors for individual review:', len(errors))

,text,word_count,label_name,tfidf_prediction_name,minilm_prediction_name,error_pattern
0,Live: Olympics day four,4,World,Sports,Sports,both_wrong
1,Google Lowers Its IPO Price Range,6,World,Business,Business,both_wrong
2,Small computers can have multiple personalitie...,7,Sci/Tech,Business,Sci/Tech,tfidf_only_wrong


All errors for individual review: 1498


## 7. Try it — Classify a *new* headline without retraining

Edit `NEW_HEADLINE` below and run the cell. **Default:** the saved TF-IDF model, so this cell works offline. Set `CHOSEN_MODEL = "minilm"` to try the pretrained embedding model (requires the `sentence-transformers` package and an Internet connection for the first weight download). Blank input should produce a helpful message, not an exception. Qualitative examples do not affect test scores.

In [7]:
NEW_HEADLINE = "Local football team wins regional championship"
CHOSEN_MODEL = "tfidf"  # "tfidf" or "minilm"
if CHOSEN_MODEL == "tfidf":
    result = predict_tfidf(NEW_HEADLINE)
elif CHOSEN_MODEL == "minilm":
    result = predict_minilm(NEW_HEADLINE)
else:
    raise ValueError("CHOSEN_MODEL must be 'tfidf' or 'minilm'")
print(result)
print('Empty-input check:', predict_tfidf('   '))

{'ok': True, 'model': 'TF-IDF + Logistic Regression', 'id': 2, 'category': 'Sports'}
Empty-input check: {'ok': False, 'message': 'Please enter a non-empty English news headline.'}


## 8. Sources, limitations, and responsible use

**Primary sources and technical references**

1. X. Zhang, J. Zhao, and Y. LeCun, “Character-level Convolutional Networks for Text Classification,” NeurIPS 2015. [Paper](https://arxiv.org/abs/1509.01626).
2. N. Reimers and I. Gurevych, “Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks,” EMNLP-IJCNLP 2019. [Paper](https://aclanthology.org/D19-1410/).
3. [AG News dataset](https://huggingface.co/datasets/sh0416/ag_news).
4. [Scikit-learn TF-IDF documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) and [Logistic Regression documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).
5. [Sentence Transformers documentation](https://www.sbert.net/).

**Limitations:** the historical news headlines contain ambiguities and sometimes unusual or debatable ground-truth labels; findings may not generalize to current news domains, other languages, full articles, or unseen topics. Our comparison is specific to the two implementations, not a claim about all language models.

**Reused resources and AI assistance:** We used the public AG News dataset, pretrained MiniLM weights, scikit-learn and sentence-transformers libraries. AI assistance supported code structuring, debugging and drafting; each team member is responsible for checking and understanding their contributions. All original examples must be personally reviewed and labelled before submission.

**Reproduction:** See `README.md` for input files, exact commands, expected outputs, package versions, and the separate training scripts.